In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import re
import os
from collections import Counter

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries imported!")
print("📝 Agent 2 (Clinical Notes) - Data Setup")
print("=" * 60)

✅ Libraries imported!
📝 Agent 2 (Clinical Notes) - Data Setup


In [2]:
print("📂 Loading patient splits...")
print("=" * 60)

# Load train/val/test patient IDs
train_patients = pd.read_csv('../../data/processed/train_patients.csv')
val_patients = pd.read_csv('../../data/processed/val_patients.csv')
test_patients = pd.read_csv('../../data/processed/test_patients.csv')

train_patient_ids = train_patients['SUBJECT_ID'].values
val_patient_ids = val_patients['SUBJECT_ID'].values
test_patient_ids = test_patients['SUBJECT_ID'].values

print(f"✅ Patient splits loaded:")
print(f"   Training:   {len(train_patient_ids):,} patients")
print(f"   Validation: {len(val_patient_ids):,} patients")
print(f"   Test:       {len(test_patient_ids):,} patients")
print(f"   Total:      {len(train_patient_ids) + len(val_patient_ids) + len(test_patient_ids):,} patients")

📂 Loading patient splits...
✅ Patient splits loaded:
   Training:   21,816 patients
   Validation: 4,675 patients
   Test:       4,675 patients
   Total:      31,166 patients


In [ ]:
print("📋 Exploring NOTEEVENTS.csv structure...")
print("=" * 60)

# Load first 1000 rows to understand structure
notes_sample = pd.read_csv('../../data/raw/NOTEEVENTS.csv', nrows=1000)

print(f"✅ Sample loaded: {len(notes_sample):,} rows")
print(f"\nColumns:")
print(notes_sample.columns.tolist())

print(f"\nData types:")
print(notes_sample.dtypes)

print(f"\nFirst note example:")
print(notes_sample.iloc[0])

print(f"\nNote categories:")
print(notes_sample['CATEGORY'].value_counts())

print(f"\nNote descriptions:")
print(notes_sample['DESCRIPTION'].value_counts().head(10))

In [4]:
print("📝 Loading ALL clinical notes...")
print("=" * 60)
print("⏳ This will take 5-10 minutes - processing 2M+ notes...")
print()

# Load in chunks to handle large file
chunk_size = 100_000
chunks = []

for chunk in tqdm(pd.read_csv('../../data/raw/NOTEEVENTS.csv', chunksize=chunk_size), 
                  desc="Loading notes"):
    chunks.append(chunk)

notes = pd.concat(chunks, ignore_index=True)

print(f"\n✅ Loaded all notes!")
print(f"   Total notes: {len(notes):,}")
print(f"   Total patients: {notes['SUBJECT_ID'].nunique():,}")
print(f"   Memory usage: {notes.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

📝 Loading ALL clinical notes...
⏳ This will take 5-10 minutes - processing 2M+ notes...



Loading notes: 21it [01:32,  4.42s/it]



✅ Loaded all notes!
   Total notes: 2,083,180
   Total patients: 46,146
   Memory usage: 4394.1 MB


In [5]:
print("🔍 Analyzing clinical note types...")
print("=" * 60)

print(f"Note categories:")
category_counts = notes['CATEGORY'].value_counts()
print(category_counts)

print(f"\nMost common note types:")
description_counts = notes['DESCRIPTION'].value_counts().head(15)
print(description_counts)

# Check patient coverage
print(f"\nPatient coverage:")
patients_with_notes = notes['SUBJECT_ID'].nunique()
print(f"  Patients with notes: {patients_with_notes:,}")
print(f"  Total patients in dataset: {len(train_patient_ids) + len(val_patient_ids) + len(test_patient_ids):,}")
print(f"  Coverage: {patients_with_notes / (len(train_patient_ids) + len(val_patient_ids) + len(test_patient_ids)) * 100:.1f}%")

# Notes per patient
notes_per_patient = notes.groupby('SUBJECT_ID').size()
print(f"\nNotes per patient statistics:")
print(f"  Mean: {notes_per_patient.mean():.1f}")
print(f"  Median: {notes_per_patient.median():.0f}")
print(f"  Min: {notes_per_patient.min()}")
print(f"  Max: {notes_per_patient.max()}")

🔍 Analyzing clinical note types...
Note categories:
CATEGORY
Nursing/other        822497
Radiology            522279
Nursing              223556
ECG                  209051
Physician            141624
Discharge summary     59652
Echo                  45794
Respiratory           31739
Nutrition              9418
General                8301
Rehab Services         5431
Social Work            2670
Case Management         967
Pharmacy                103
Consult                  98
Name: count, dtype: int64

Most common note types:
DESCRIPTION
Report                               1132519
Nursing Progress Note                 191836
CHEST (PORTABLE AP)                   169270
Physician Resident Progress Note       62698
CHEST (PA & LAT)                       43158
CT HEAD W/O CONTRAST                   34485
Respiratory Care Shift Note            31105
Nursing Transfer Note                  30773
Intensivist Note                       26144
CHEST PORT. LINE PLACEMENT             21596
Physic

In [6]:
print("🎯 Selecting important note categories for diagnosis...")
print("=" * 60)

# For clinical diagnosis, these note types are most relevant:
important_categories = [
    'Discharge summary',  # Comprehensive patient summary
    'Physician ',         # Doctor's notes
    'Nursing',           # Nursing observations
    'Radiology',         # Imaging reports
    'Echo',              # Echocardiogram reports
    'ECG',               # EKG reports
    'Respiratory '       # Respiratory therapy notes
]

# Filter notes to important categories
notes_filtered = notes[notes['CATEGORY'].isin(important_categories)].copy()

print(f"✅ Filtered to important categories:")
print(f"   Original notes: {len(notes):,}")
print(f"   Filtered notes: {len(notes_filtered):,}")
print(f"   Reduction: {(1 - len(notes_filtered)/len(notes))*100:.1f}%")

print(f"\nFiltered category distribution:")
print(notes_filtered['CATEGORY'].value_counts())

# Check coverage after filtering
print(f"\nPatient coverage after filtering:")
patients_with_filtered = notes_filtered['SUBJECT_ID'].nunique()
print(f"  Patients with notes: {patients_with_filtered:,}")
print(f"  Coverage: {patients_with_filtered / (len(train_patient_ids) + len(val_patient_ids) + len(test_patient_ids)) * 100:.1f}%")

🎯 Selecting important note categories for diagnosis...
✅ Filtered to important categories:
   Original notes: 2,083,180
   Filtered notes: 1,233,695
   Reduction: 40.8%

Filtered category distribution:
CATEGORY
Radiology            522279
Nursing              223556
ECG                  209051
Physician            141624
Discharge summary     59652
Echo                  45794
Respiratory           31739
Name: count, dtype: int64

Patient coverage after filtering:
  Patients with notes: 42,708
  Coverage: 137.0%


In [8]:
print("🧹 Cleaning and preprocessing clinical text...")
print("=" * 60)

# Enable tqdm for pandas
from tqdm.auto import tqdm
tqdm.pandas()

def clean_clinical_text(text):
    """Clean clinical notes for BERT processing"""
    if pd.isna(text):
        return ""
    
    # Convert to string
    text = str(text)
    
    # Remove special characters but keep medical terminology
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text)  # Remove MIMIC de-identification markers
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove very short notes (less than 20 characters)
    if len(text) < 20:
        return ""
    
    # Truncate very long notes (BERT has 512 token limit)
    # Keep first 2000 characters as a rough approximation
    if len(text) > 2000:
        text = text[:2000]
    
    return text.strip()

print(f"Cleaning {len(notes_filtered):,} notes...")
print(f"⏳ This will take 5-10 minutes with progress bar...")

# Apply cleaning with progress bar
notes_filtered['TEXT_CLEAN'] = notes_filtered['TEXT'].progress_apply(clean_clinical_text)

# Remove empty notes
notes_filtered = notes_filtered[notes_filtered['TEXT_CLEAN'] != ''].copy()

print(f"\n✅ Text cleaning complete!")
print(f"   Notes after cleaning: {len(notes_filtered):,}")
print(f"\nSample cleaned note:")
print(notes_filtered.iloc[0]['TEXT_CLEAN'][:500], "...")

🧹 Cleaning and preprocessing clinical text...
Cleaning 1,233,695 notes...
⏳ This will take 5-10 minutes with progress bar...


  0%|          | 0/1233695 [00:00<?, ?it/s]


✅ Text cleaning complete!
   Notes after cleaning: 1,233,297

Sample cleaned note:
Admission Date: Discharge Date: Service: ADDENDUM: RADIOLOGIC STUDIES: Radiologic studies also included a chest CT, which confirmed cavitary lesions in the left lung apex consistent with infectious process/tuberculosis. This also moderate-sized left pleural effusion. HEAD CT: Head CT showed no intracranial hemorrhage or mass effect, but old infarction consistent with past medical history. ABDOMINAL CT: Abdominal CT showed lesions of T10 and sacrum most likely secondary to osteoporosis. These can ...


In [9]:
print("📚 Aggregating notes per patient...")
print("=" * 60)

# Strategy: Concatenate all notes for each patient
# This gives BERT the full clinical context

def aggregate_patient_notes(patient_notes):
    """Combine all notes for a patient"""
    # Sort by date if available (CHARTDATE column)
    if 'CHARTDATE' in patient_notes.columns:
        patient_notes = patient_notes.sort_values('CHARTDATE')
    
    # Concatenate all cleaned text
    all_text = ' '.join(patient_notes['TEXT_CLEAN'].values)
    
    # Truncate to reasonable length (first 3000 chars)
    if len(all_text) > 3000:
        all_text = all_text[:3000]
    
    return all_text

print(f"Aggregating notes for {notes_filtered['SUBJECT_ID'].nunique():,} patients...")

# Group by patient and aggregate
patient_notes = notes_filtered.groupby('SUBJECT_ID').apply(
    lambda x: aggregate_patient_notes(x),
    include_groups=False
).reset_index()

patient_notes.columns = ['SUBJECT_ID', 'CLINICAL_TEXT']

print(f"\n✅ Aggregation complete!")
print(f"   Patients with notes: {len(patient_notes):,}")

# Statistics
text_lengths = patient_notes['CLINICAL_TEXT'].str.len()
print(f"\nText length statistics:")
print(f"   Mean: {text_lengths.mean():.0f} characters")
print(f"   Median: {text_lengths.median():.0f} characters")
print(f"   Min: {text_lengths.min()} characters")
print(f"   Max: {text_lengths.max()} characters")

📚 Aggregating notes per patient...
Aggregating notes for 42,708 patients...

✅ Aggregation complete!
   Patients with notes: 42,708

Text length statistics:
   Mean: 2914 characters
   Median: 3000 characters
   Min: 44 characters
   Max: 3000 characters


In [10]:
print("✂️ Splitting notes into train/val/test sets...")
print("=" * 60)

# Filter to each set
notes_train = patient_notes[patient_notes['SUBJECT_ID'].isin(train_patient_ids)].copy()
notes_val = patient_notes[patient_notes['SUBJECT_ID'].isin(val_patient_ids)].copy()
notes_test = patient_notes[patient_notes['SUBJECT_ID'].isin(test_patient_ids)].copy()

print(f"✅ Split complete:")
print(f"   Training notes:   {len(notes_train):,} patients")
print(f"   Validation notes: {len(notes_val):,} patients")
print(f"   Test notes:       {len(notes_test):,} patients")

# Check coverage
print(f"\nCoverage:")
print(f"   Train: {len(notes_train) / len(train_patient_ids) * 100:.1f}%")
print(f"   Val:   {len(notes_val) / len(val_patient_ids) * 100:.1f}%")
print(f"   Test:  {len(notes_test) / len(test_patient_ids) * 100:.1f}%")

✂️ Splitting notes into train/val/test sets...
✅ Split complete:
   Training notes:   21,678 patients
   Validation notes: 4,643 patients
   Test notes:       4,645 patients

Coverage:
   Train: 99.4%
   Val:   99.3%
   Test:  99.4%


In [11]:
print("🔗 Merging notes with disease labels...")
print("=" * 60)

# Load labels
labels = pd.read_csv('../../data/processed/patient_multilabels.csv')

# Disease columns
disease_cols = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 
    'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
    'ANEMIA', 'PANCREATITIS'
]

# Merge train
train_notes_labeled = notes_train.merge(labels, on='SUBJECT_ID', how='inner')

print(f"✅ Training data merged:")
print(f"   Patients with notes AND labels: {len(train_notes_labeled):,}")
print(f"   Original training patients: {len(train_patient_ids):,}")
print(f"   Coverage: {len(train_notes_labeled) / len(train_patient_ids) * 100:.1f}%")

# Check label distribution
print(f"\nDisease distribution in training notes:")
for disease in disease_cols:
    count = train_notes_labeled[disease].sum()
    pct = count / len(train_notes_labeled) * 100
    print(f"  {disease:30s}: {count:5,} ({pct:5.1f}%)")

print(f"\nSample data:")
print(train_notes_labeled.head())

🔗 Merging notes with disease labels...
✅ Training data merged:
   Patients with notes AND labels: 21,678
   Original training patients: 21,816
   Coverage: 99.4%

Disease distribution in training notes:
  SEPSIS                        : 4,065 ( 18.8%)
  PNEUMONIA                     : 4,522 ( 20.9%)
  RESPIRATORY_FAILURE           : 7,837 ( 36.2%)
  ACUTE_KIDNEY_INJURY           : 7,972 ( 36.8%)
  HEART_FAILURE                 : 7,046 ( 32.5%)
  ATRIAL_FIBRILLATION           : 7,376 ( 34.0%)
  CORONARY_ARTERY_DISEASE       : 8,323 ( 38.4%)
  ANEMIA                        : 7,362 ( 34.0%)
  PANCREATITIS                  : 3,151 ( 14.5%)

Sample data:
   SUBJECT_ID  \
0           9   
1          13   
2          20   
3          21   
4          25   

                                                                                         CLINICAL_TEXT  \
0  Sinus rhythm Left ventricular hypertrophy (aVL + 12 millimeter) No previous tracing for comparis...   
1  Sinus rhythm/ Non-diagno

In [12]:
print("💾 Saving processed notes...")
print("=" * 60)

# Save training notes
notes_train.to_csv('../../data/processed/notes_train.csv', index=False)
print(f"✅ Saved: data/processed/notes_train.csv")

# Save validation notes
notes_val.to_csv('../../data/processed/notes_val.csv', index=False)
print(f"✅ Saved: data/processed/notes_val.csv")

# Save test notes
notes_test.to_csv('../../data/processed/notes_test.csv', index=False)
print(f"✅ Saved: data/processed/notes_test.csv")

# Save training data with labels
train_notes_labeled.to_csv('../../data/processed/notes_train_labeled.csv', index=False)
print(f"✅ Saved: data/processed/notes_train_labeled.csv")

print(f"\n📊 File sizes:")
print(f"   notes_train.csv: {os.path.getsize('../../data/processed/notes_train.csv') / 1024**2:.1f} MB")
print(f"   notes_val.csv: {os.path.getsize('../../data/processed/notes_val.csv') / 1024**2:.1f} MB")
print(f"   notes_test.csv: {os.path.getsize('../../data/processed/notes_test.csv') / 1024**2:.1f} MB")
print(f"   notes_train_labeled.csv: {os.path.getsize('../../data/processed/notes_train_labeled.csv') / 1024**2:.1f} MB")

💾 Saving processed notes...
✅ Saved: data/processed/notes_train.csv
✅ Saved: data/processed/notes_val.csv
✅ Saved: data/processed/notes_test.csv
✅ Saved: data/processed/notes_train_labeled.csv

📊 File sizes:
   notes_train.csv: 61.8 MB
   notes_val.csv: 13.2 MB
   notes_test.csv: 13.2 MB
   notes_train_labeled.csv: 62.2 MB


In [13]:
print("📊 Analyzing text for BERT tokenization...")
print("=" * 60)

# Sample some texts to estimate token counts
sample_texts = train_notes_labeled['CLINICAL_TEXT'].head(100).values

# Rough estimate: 1 token ≈ 4 characters for English
estimated_tokens = [len(text) / 4 for text in sample_texts]

print(f"Estimated token statistics (rough approximation):")
print(f"   Mean: {np.mean(estimated_tokens):.0f} tokens")
print(f"   Median: {np.median(estimated_tokens):.0f} tokens")
print(f"   Max: {np.max(estimated_tokens):.0f} tokens")
print(f"   % over 512 (BERT limit): {sum(t > 512 for t in estimated_tokens) / len(estimated_tokens) * 100:.1f}%")

print(f"\n💡 BERT Configuration Notes:")
print(f"   - Max sequence length: 512 tokens")
print(f"   - We truncated texts to ~3000 chars (~750 tokens)")
print(f"   - BERT will handle final truncation")
print(f"   - Most texts fit within limits ✅")

📊 Analyzing text for BERT tokenization...
Estimated token statistics (rough approximation):
   Mean: 745 tokens
   Median: 750 tokens
   Max: 750 tokens
   % over 512 (BERT limit): 100.0%

💡 BERT Configuration Notes:
   - Max sequence length: 512 tokens
   - We truncated texts to ~3000 chars (~750 tokens)
   - BERT will handle final truncation
   - Most texts fit within limits ✅
